# Proyecto final: Monitorización de la calidad del aire en ciudades inteligencias

Autor: Adrián Robles Arques

Fuentes de los datos:
* Datos de los diferentes sensores de contaminantes: 
    * Viver: https://aqicn.org/historical/#!city:spain/valencia/viver
    * Quart de Poblet: https://aqicn.org/historical/#!city:spain/valencia/quart-de-poblet
    * Pista de Silla: https://aqicn.org/historical/#!city:spain/valencia/valencia/pista-de-silla
    * Molí del Sol: https://aqicn.org/historical/#!city:spain/valencia/valencia/moli-del-sol
    * Politècnic: https://aqicn.org/historical/#!city:spain/valencia/valencia/politecnic

* Datos meteorológicos de Valencia: 
    * 2024: https://meteostat.net/es/station/08285?t=2024-01-01/2024-12-31
    * 2025: https://meteostat.net/es/station/08285?t=2025-01-01/2025-06-30

## Fase 1: Carga y limpieza de los datos

En esta fase vamos a realizar una primera inspección de los datos, tanto de los diferentes sensores en tierra donde se evalúa el índice de calidad del aire para cada uno de los contaminantes considerados, así como los datos meteorológicos proporcionados por Meteosat para la ciudad de Valencia.

De este modo, para evaluar cada uno de los índices individuales tomaremos en consideración no solo valores previos, como sería usual en una serie temporal, si no también enriquecer esta fuente de datos con información meteorológica.

In [2]:
# Importamos librerías necesarias
import numpy as np
import pyspark as ps
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, FloatType, DateType, IntegerType
from pyspark.sql.window import Window

In [3]:
# Establecemos las variables de entorno necesarias
import os
os.environ['HADOOP_HOME'] = 'C:\\Hadoop\\hadoop-3.3.1'

In [4]:
print(ps.__version__)

3.5.4


In [5]:
# Creamos la sesión de Spark
# Creamos una sesion de Spark
spark = ps.sql.SparkSession.builder \
    .appName("Grafos en Spark") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

In [6]:
# Vamos a crear un DataFrame para los datos climáticos

# Importamos los datos climáticos
datos_clima_2024 = 'C:\\Users\\demad\\Desktop\\Test\\DataScienceIEBS\\Bloque 7\\Spark\\Proyecto final\\Datos calidad aire\\meteosat_val_2024.csv'
df_clima = spark.read.csv(datos_clima_2024, header=True, inferSchema=True)

# Visualizamos un fragmento del DataFrame
df_clima.show(5)

+-------------------+----+----+----+----+----+----+----+----+------+----+
|               date|tavg|tmin|tmax|prcp|snow|wdir|wspd|wpgt|  pres|tsun|
+-------------------+----+----+----+----+----+----+----+----+------+----+
|2024-01-01 00:00:00|11.1| 9.8|19.0| 0.0|NULL|NULL|11.8|NULL|1019.7|NULL|
|2024-01-02 00:00:00|13.0| 7.8|18.9| 0.0|NULL|NULL|24.3|NULL|1019.8|NULL|
|2024-01-03 00:00:00|17.4|15.0|23.0| 0.0|NULL|NULL|26.4|NULL|1017.0|NULL|
|2024-01-04 00:00:00|14.8|14.5|20.6| 0.6|NULL|NULL|12.1|NULL|1014.8|NULL|
|2024-01-05 00:00:00|13.1|12.4|18.4| 0.0|NULL|NULL|22.7|NULL|1006.0|NULL|
+-------------------+----+----+----+----+----+----+----+----+------+----+
only showing top 5 rows



Aquí vemos que los datos climáticos que tenemos son:
* Temperatura promedio (C)
* Temperatura mínima (C)
* Temperatura máxima (C)
* Precipitación acumulada (mm)
* Profundidad de la nieve
* Dirección del viento
* Velocidad del viento (km/h)
* Ráfaga de viento
* Presión del aire (hPa)
* Duración del sol

Como vemos, varias de las columnas no son necesarios o están vacías, por lo que vamos a quedarnos solo con las que nos interesan.
* Temperatura promedio (C)
* Temperatura mínima (C)
* Temperatura máxima (C)
* Precipitación acumulada (mm)
* Velocidad del viento (km/h)
* Presión del aire (hPa)

In [7]:
# Vamos a modificar el DataFrame para que tenga las columnas adecuadas
df_clima = df_clima.drop('snow', 'wdir', 'wpgt', 'tsun')

In [8]:
# También vamos a renombrar las columnas para que sean más descriptivas
df_clima = df_clima.withColumnRenamed('date', 'fecha') \
    .withColumnRenamed('time', 'hora') \
    .withColumnRenamed('tavg', 'temperatura media') \
    .withColumnRenamed('tmin', 'temperatura mínima') \
    .withColumnRenamed('tmax', 'temperatura máxima') \
    .withColumnRenamed('prcp', 'precipitación') \
    .withColumnRenamed('wspd', 'velocidad del viento') \
    .withColumnRenamed('pres', 'presión atmosférica')
    
# Convertimos la columna de fecha a tipo Date
df_clima = df_clima.withColumn('fecha', F.to_date(F.col('fecha'), 'yyyy-MM-dd'))

# Mostramos el esquema del DataFrame para verificar los cambios
df_clima.printSchema()

root
 |-- fecha: date (nullable = true)
 |-- temperatura media: double (nullable = true)
 |-- temperatura mínima: double (nullable = true)
 |-- temperatura máxima: double (nullable = true)
 |-- precipitación: double (nullable = true)
 |-- velocidad del viento: double (nullable = true)
 |-- presión atmosférica: double (nullable = true)



In [9]:
# Mostramos nuevamente el DataFrame para ver los cambios
df_clima.show(5)

+----------+-----------------+------------------+------------------+-------------+--------------------+-------------------+
|     fecha|temperatura media|temperatura mínima|temperatura máxima|precipitación|velocidad del viento|presión atmosférica|
+----------+-----------------+------------------+------------------+-------------+--------------------+-------------------+
|2024-01-01|             11.1|               9.8|              19.0|          0.0|                11.8|             1019.7|
|2024-01-02|             13.0|               7.8|              18.9|          0.0|                24.3|             1019.8|
|2024-01-03|             17.4|              15.0|              23.0|          0.0|                26.4|             1017.0|
|2024-01-04|             14.8|              14.5|              20.6|          0.6|                12.1|             1014.8|
|2024-01-05|             13.1|              12.4|              18.4|          0.0|                22.7|             1006.0|
+-------

In [10]:
# Creamos una función que replica las transformaciones que se han hecho en el DataFrame de clima, para aprovecharla en el pipeline
def ajustar_clima(df: DataFrame) -> DataFrame:
    """
    Ajusta el DataFrame de clima para que tenga las columnas adecuadas y los tipos de datos correctos.
    """
    df = df.drop('snow', 'wdir', 'wpgt', 'tsun')
    df = df.withColumnRenamed('date', 'fecha') \
        .withColumnRenamed('time', 'hora') \
        .withColumnRenamed('tavg', 'temperatura media') \
        .withColumnRenamed('tmin', 'temperatura mínima') \
        .withColumnRenamed('tmax', 'temperatura máxima') \
        .withColumnRenamed('prcp', 'precipitación') \
        .withColumnRenamed('wspd', 'velocidad del viento') \
        .withColumnRenamed('pres', 'presión atmosférica')
    
    df = df.withColumn('fecha', F.to_date(F.col('fecha'), 'yyyy-MM-dd'))
    
    return df

In [11]:
# Vamos a cargar los datos de calidad del aire
sensores = {
    'datos viver': 'C:\\Users\\demad\\Desktop\\Test\\DataScienceIEBS\\Bloque 7\\Spark\\Proyecto final\\Datos calidad aire\\sensor viver.csv',
    'datos moli del sol': 'C:\\Users\\demad\\Desktop\\Test\\DataScienceIEBS\\Bloque 7\\Spark\\Proyecto final\\Datos calidad aire\\sensor moli del sol.csv',
    'datos pista de silla': 'C:\\Users\\demad\\Desktop\\Test\\DataScienceIEBS\\Bloque 7\\Spark\\Proyecto final\\Datos calidad aire\\sensor pista de silla.csv',
    'datos politècnic': 'C:\\Users\\demad\\Desktop\\Test\\DataScienceIEBS\\Bloque 7\\Spark\\Proyecto final\\Datos calidad aire\\sensor politecnic.csv',
    'datos quart de poblet': 'C:\\Users\\demad\\Desktop\\Test\\DataScienceIEBS\\Bloque 7\\Spark\\Proyecto final\\Datos calidad aire\\sensor quart de poblet.csv'
}

# Cargamos los datos de cada sensor y los almacenamos en un diccionario
sensor_viver = spark.read.csv(sensores['datos viver'], header=True, inferSchema=True)
sensor_moli_del_sol = spark.read.csv(sensores['datos moli del sol'], header=True, inferSchema=True)
sensor_pista_de_silla = spark.read.csv(sensores['datos pista de silla'], header=True, inferSchema=True)
sensor_politecnic = spark.read.csv(sensores['datos politècnic'], header=True, inferSchema=True)
sensor_quart_de_poblet = spark.read.csv(sensores['datos quart de poblet'], header=True, inferSchema=True)


# Vamos a almacenarlos en un diccionario para facilitar su manejo
datos_sensores = {
    'viver': sensor_viver,
    'moli del sol': sensor_moli_del_sol,
    'pista de silla': sensor_pista_de_silla,
    'politecnic': sensor_politecnic,
    'quart de poblet': sensor_quart_de_poblet
}

In [12]:
# Vamos a explorar los datos de un sensor como ejemplo
datos_sensores['viver'].show(5)

+--------+-----+-----+---+----+----+---+
|    date| pm25| pm10| o3| no2| so2| co|
+--------+-----+-----+---+----+----+---+
|2025/6/1|   18|    6| 40|   4|   1|   |
|2025/6/2|   15|    7| 41|   2|   1|   |
|2025/6/3|   19|    9| 43|   3|   1|   |
|2025/6/4|   32|   12| 45|   3|   1|   |
|2025/6/5|   38|   12| 48|   3|   1|   |
+--------+-----+-----+---+----+----+---+
only showing top 5 rows



In [13]:
# Mostramos el esquema del DataFrame del sensor
datos_sensores['viver'].printSchema()

root
 |-- date: string (nullable = true)
 |--  pm25: string (nullable = true)
 |--  pm10: string (nullable = true)
 |--  o3: string (nullable = true)
 |--  no2: string (nullable = true)
 |--  so2: string (nullable = true)
 |--  co: string (nullable = true)



In [14]:
# Normaliza el formato de la fecha a yyyy/MM/dd (dos dígitos para mes y día)
# Divide la fecha en partes y añade ceros a la izquierda si es necesario
sensor_df = sensor_viver.withColumn('year', F.split(F.col('date'), '/').getItem(0))
sensor_df = sensor_df.withColumn('month', F.lpad(F.split(F.col('date'), '/').getItem(1), 2, '0'))
sensor_df = sensor_df.withColumn('day', F.lpad(F.split(F.col('date'), '/').getItem(2), 2, '0'))

# Une las partes en formato yyyy/MM/dd
sensor_df = sensor_df.withColumn('date', F.concat_ws('-', F.col('year'), F.col('month'), F.col('day')))

# Eliminamos las columnas auxiliares
sensor_df = sensor_df.drop('year', 'month', 'day')

# Convertimos la columna de fecha a tipo Date
sensor_df = sensor_df.withColumn('date', F.to_date(F.col('date'), 'yyyy-MM-dd'))

# Mostramos los datos
sensor_df.show(5)

+----------+-----+-----+---+----+----+---+
|      date| pm25| pm10| o3| no2| so2| co|
+----------+-----+-----+---+----+----+---+
|2025-06-01|   18|    6| 40|   4|   1|   |
|2025-06-02|   15|    7| 41|   2|   1|   |
|2025-06-03|   19|    9| 43|   3|   1|   |
|2025-06-04|   32|   12| 45|   3|   1|   |
|2025-06-05|   38|   12| 48|   3|   1|   |
+----------+-----+-----+---+----+----+---+
only showing top 5 rows



In [15]:
# Muestra el esquema
sensor_df.printSchema()

root
 |-- date: date (nullable = true)
 |--  pm25: string (nullable = true)
 |--  pm10: string (nullable = true)
 |--  o3: string (nullable = true)
 |--  no2: string (nullable = true)
 |--  so2: string (nullable = true)
 |--  co: string (nullable = true)



Como puede verse, los datos de los sensores ya están convertidos para mostrar el Índice calculado para cada uno de los contaminantes. También se aprecia que hay algunos valores que están en blanco, por lo que habrá que imputarlos. Además, los datos se han cargado en formato cadena, lo que requerirá una transformación. En primer lugar intenté cargar los datos introduciendo directamente el esquema deseado, pero en ese caso no leía ningún dato y solo generaba Dataframes con todas las líneas vacías, por problemas con el tipado de las fechas y los espacios en blanco. De modo que vamos a hacer el cambio a posteriori.

In [16]:
# Antes de convertir, tenemos que asegurarnos de que los datos están en el formato correcto
# Para ello, debemos eliminar los espacios en blanco en las cadenas de texto, y transformar huecos en valores nulos

# Generamos una función para realizar el proceso deseado

def sensor_adjustments(sensor_df: DataFrame) -> DataFrame:
    
    # Normaliza el formato de la fecha a yyyy/MM/dd (dos dígitos para mes y día)

    # Divide la fecha en partes y añade ceros a la izquierda si es necesario
    sensor_df = sensor_df.withColumn('year', F.split(F.col('date'), '/').getItem(0))
    sensor_df = sensor_df.withColumn('month', F.lpad(F.split(F.col('date'), '/').getItem(1), 2, '0'))
    sensor_df = sensor_df.withColumn('day', F.lpad(F.split(F.col('date'), '/').getItem(2), 2, '0'))

    # Une las partes en formato yyyy/MM/dd
    sensor_df = sensor_df.withColumn('date', F.concat_ws('-', F.col('year'), F.col('month'), F.col('day')))

    # Eliminamos las columnas auxiliares
    sensor_df = sensor_df.drop('year', 'month', 'day')
    
    # Limpia los nombres de las columnas
    for col in sensor_df.columns:
        sensor_df = sensor_df.withColumnRenamed(col, col.strip())
        
    # Eliminamos espacios en blanco al principio y al final de cada columna
    sensor_df = sensor_df.select([F.trim(F.col(c)).alias(c) for c in sensor_df.columns])
    
    # Convertimos strings vacíos a nulos
    sensor_df = sensor_df.select([F.when(F.col(c) == '', None).otherwise(F.col(c)).alias(c) for c in sensor_df.columns])
    
    # Convertimos las columnas a los tipos adecuados
    sensor_df = sensor_df.withColumn('date', F.to_date(F.col('date'), 'yyyy-MM-dd')) \
                        .withColumn('pm25', F.col('pm25').cast(IntegerType())) \
                        .withColumn('pm10', F.col('pm10').cast(IntegerType())) \
                        .withColumn('o3', F.col('no2').cast(IntegerType())) \
                        .withColumn('no2', F.col('no2').cast(IntegerType())) \
                        .withColumn('so2', F.col('so2').cast(IntegerType())) \
                        .withColumn('co', F.col('co').cast(IntegerType()))
    
    return sensor_df

# Aplicamos la función a cada DataFrame de sensor
for key, sensor_df in datos_sensores.items():
    datos_sensores[key] = sensor_adjustments(sensor_df)
    
# Mostramos el esquema del DataFrame del sensor
datos_sensores['viver'].printSchema()


root
 |-- date: date (nullable = true)
 |-- pm25: integer (nullable = true)
 |-- pm10: integer (nullable = true)
 |-- o3: integer (nullable = true)
 |-- no2: integer (nullable = true)
 |-- so2: integer (nullable = true)
 |-- co: integer (nullable = true)



In [17]:
# Vamos a mostrar datos de un sensor para ver cómo ha quedado
datos_sensores['viver'].show(5)

+----------+----+----+---+---+---+----+
|      date|pm25|pm10| o3|no2|so2|  co|
+----------+----+----+---+---+---+----+
|2025-06-01|  18|   6|  4|  4|  1|NULL|
|2025-06-02|  15|   7|  2|  2|  1|NULL|
|2025-06-03|  19|   9|  3|  3|  1|NULL|
|2025-06-04|  32|  12|  3|  3|  1|NULL|
|2025-06-05|  38|  12|  3|  3|  1|NULL|
+----------+----+----+---+---+---+----+
only showing top 5 rows



Con los datos ya en el formato deseado, tal como puede verse en el esquema mostrado a modo de ejemplo, vamos a extraer los datos de los años deseados. Cada archivo CSV contiene datos de todos los años disponibles, empezando por el más reciente. Vamos a extraer los datos de 2024 para entrenamiento y los de 2025 para validación.

In [18]:
# Creamos una función que filtre los datos por un año dado
def datos_anuales(sensor_df: DataFrame, year: int) -> DataFrame:
    """
    Filtra los datos del sensor por un año específico.
    
    INPUTS: 
    - sensor_df: DataFrame del sensor.
    - year: Año a filtrar.
    
    RETURNS:
    - DataFrame filtrado por el año especificado.
    """
    return sensor_df.filter(F.year(F.col('date')) == year)

In [19]:
# Seleccionamos los datos del año 2024 para entrenamiento de todos los sensores
datos_2024 = {key: datos_anuales(sensor_df, 2024) for key, sensor_df in datos_sensores.items()}

In [20]:
# Mostramos los datos filtrados de un sensor
datos_2024['viver'].show(5)

+----------+----+----+----+----+----+----+
|      date|pm25|pm10|  o3| no2| so2|  co|
+----------+----+----+----+----+----+----+
|2024-10-01|  14|NULL|NULL|NULL|NULL|NULL|
|2024-10-03|   7|   3|   2|   2|   1|NULL|
|2024-10-04|   9|   5|   3|   3|   1|NULL|
|2024-10-05|  15|  18|   1|   1|   1|NULL|
|2024-10-06|  42|  15|   7|   7|   1|NULL|
+----------+----+----+----+----+----+----+
only showing top 5 rows



Extraidos los datos para entrenamiento, vamos a imputar los posibles valores faltantes a partir de datos estadísticos de cada columna.

In [21]:
# Para los datos de sensores de 2024, vamos a imputar los valores nulos con la media del valor previo y posterior

def imputar_valores(sensor_df: DataFrame, get_int = True, date_column = 'date') -> DataFrame:
    """
    Imputa los valores nulos en el DataFrame del sensor con la media del valor previo y posterior.
    Si alguno de los valores es nulo, se considera como 0 para el cálculo de la media.
    
    INPUTS:
    - sensor_df: DataFrame del sensor.
    
    RETURNS:
    - DataFrame con los valores nulos imputados.
    """
    for col in sensor_df.columns:
        if col != date_column:  # No procesamos la columna de fecha
            window = Window.orderBy(date_column) # Generamos una ventana ordenada por fecha
            # Imputamos los valores nulos con la media del valor previo y posterior
            sensor_df = sensor_df.withColumn(
                col,
                F.when(F.col(col).isNull(), # Para valores nulos, procesamos el valor previo y posterior
                    ( # Vamos a verificar que ninguno de los dos valores sea a su vez nulo
                        F.coalesce(F.lag(F.col(col), 1).over(window), F.lit(0)) + 
                        F.coalesce(F.lead(F.col(col), 1).over(window), F.lit(0))) / 2 # Calculamos la media
                ).otherwise(F.col(col))
            )
            
            # Convertimos los valores a enteros, ya que los datos de los sensores son enteros
            if get_int:
                sensor_df = sensor_df.withColumn(col, F.col(col).cast(IntegerType()))
            
    return sensor_df
    

In [22]:
# Vamos a emplear la función de imputación en los datos de 2024
datos_2024_imputados = {key: imputar_valores(sensor_df) for key, sensor_df in datos_2024.items()}


In [23]:
# Vamos a mostrar nuevamente los datos
datos_2024_imputados['viver'].show(10)

+----------+----+----+---+---+---+---+
|      date|pm25|pm10| o3|no2|so2| co|
+----------+----+----+---+---+---+---+
|2024-01-01|  10|   8|  1|  1|  2|  0|
|2024-01-02|  14|   2|  1|  1|  1|  0|
|2024-01-03|   7|   7|  3|  3|  1|  0|
|2024-01-04|  28|   2|  1|  1|  1|  0|
|2024-01-05|   9|   4|  0|  0|  1|  0|
|2024-01-06|   7|   1|  0|  0|  1|  0|
|2024-01-07|  12|   2|  0|  0|  1|  0|
|2024-01-08|  17|   5|  1|  1|  1|  0|
|2024-01-09|  16|   9|  1|  1|  1|  0|
|2024-01-10|  34|   2|  1|  1|  1|  0|
+----------+----+----+---+---+---+---+
only showing top 10 rows



In [24]:
# Vamos a comprobar si tenemos algún dato nulo más en los datos de entrenamiento

for datos in datos_2024_imputados.values():
    if datos.filter(F.col('pm25').isNull() | F.col('pm10').isNull() | F.col('o3').isNull() | 
                    F.col('no2').isNull() | F.col('so2').isNull() | F.col('co').isNull()).count() > 0:
        print("Hay datos nulos en el DataFrame del sensor.")
    else:
        print("No hay datos nulos en el DataFrame del sensor.")
        
if df_clima.filter(
    F.col('temperatura media').isNull() | 
    F.col('temperatura mínima').isNull() | 
    F.col('temperatura máxima').isNull() | 
    F.col('precipitación').isNull() | 
    F.col('velocidad del viento').isNull() | 
    F.col('presión atmosférica').isNull()
).count() > 0:
    print("Hay datos nulos en el DataFrame del clima.")
else:
    print("No hay datos nulos en el DataFrame del clima.")

No hay datos nulos en el DataFrame del sensor.
No hay datos nulos en el DataFrame del sensor.
No hay datos nulos en el DataFrame del sensor.
No hay datos nulos en el DataFrame del sensor.
No hay datos nulos en el DataFrame del sensor.
Hay datos nulos en el DataFrame del clima.


In [25]:
df_clima.filter(
    F.col('temperatura media').isNull() | 
    F.col('temperatura mínima').isNull() | 
    F.col('temperatura máxima').isNull() | 
    F.col('precipitación').isNull() | 
    F.col('velocidad del viento').isNull() | 
    F.col('presión atmosférica').isNull()
).show()

+----------+-----------------+------------------+------------------+-------------+--------------------+-------------------+
|     fecha|temperatura media|temperatura mínima|temperatura máxima|precipitación|velocidad del viento|presión atmosférica|
+----------+-----------------+------------------+------------------+-------------+--------------------+-------------------+
|2024-08-28|             NULL|              26.1|              30.2|          0.0|                NULL|               NULL|
|2024-08-29|             NULL|              24.7|              31.0|          0.0|                NULL|               NULL|
|2024-08-30|             NULL|              25.0|              29.2|         NULL|                NULL|               NULL|
|2024-08-31|             NULL|              24.2|              30.8|          0.0|                NULL|               NULL|
|2024-09-01|             NULL|              24.2|              30.8|         NULL|                NULL|               NULL|
|2024-09

In [26]:
# La temperatura media podemos calcularla para los datos faltantes a partir de la temperatura mínima y máxima
df_clima = df_clima.withColumn(
    'temperatura media',
    F.when(F.col('temperatura media').isNull(),
            (F.col('temperatura mínima') + F.col('temperatura máxima')) / 2
    ).otherwise(F.col('temperatura media'))
)

In [27]:
df_clima.filter(
    F.col('temperatura media').isNull() | 
    F.col('temperatura mínima').isNull() | 
    F.col('temperatura máxima').isNull() | 
    F.col('precipitación').isNull() | 
    F.col('velocidad del viento').isNull() | 
    F.col('presión atmosférica').isNull()
).show()

+----------+------------------+------------------+------------------+-------------+--------------------+-------------------+
|     fecha| temperatura media|temperatura mínima|temperatura máxima|precipitación|velocidad del viento|presión atmosférica|
+----------+------------------+------------------+------------------+-------------+--------------------+-------------------+
|2024-08-28|             28.15|              26.1|              30.2|          0.0|                NULL|               NULL|
|2024-08-29|             27.85|              24.7|              31.0|          0.0|                NULL|               NULL|
|2024-08-30|              27.1|              25.0|              29.2|         NULL|                NULL|               NULL|
|2024-08-31|              27.5|              24.2|              30.8|          0.0|                NULL|               NULL|
|2024-09-01|              27.5|              24.2|              30.8|         NULL|                NULL|               NULL|


In [28]:
# Vamos a imputar el resto de datos usando la función que hemos creado previamente
df_clima_imputado = imputar_valores(df_clima, get_int=False, date_column='fecha')

In [29]:
df_clima_imputado.show(20)

+----------+-----------------+------------------+------------------+-------------+--------------------+-------------------+
|     fecha|temperatura media|temperatura mínima|temperatura máxima|precipitación|velocidad del viento|presión atmosférica|
+----------+-----------------+------------------+------------------+-------------+--------------------+-------------------+
|2024-01-01|             11.1|               9.8|              19.0|          0.0|                11.8|             1019.7|
|2024-01-02|             13.0|               7.8|              18.9|          0.0|                24.3|             1019.8|
|2024-01-03|             17.4|              15.0|              23.0|          0.0|                26.4|             1017.0|
|2024-01-04|             14.8|              14.5|              20.6|          0.6|                12.1|             1014.8|
|2024-01-05|             13.1|              12.4|              18.4|          0.0|                22.7|             1006.0|
|2024-01

In [30]:
if df_clima_imputado.filter(
    F.col('temperatura media').isNull() | 
    F.col('temperatura mínima').isNull() | 
    F.col('temperatura máxima').isNull() | 
    F.col('precipitación').isNull() | 
    F.col('velocidad del viento').isNull() | 
    F.col('presión atmosférica').isNull()
).count() > 1:
    print("Hay datos nulos en el DataFrame del clima.")
else:
    print("No hay datos nulos en el DataFrame del clima.")

No hay datos nulos en el DataFrame del clima.


Una vez hemos verificado que no tenemos datos nulos o inválidos en los datos de entrenamiento, vamos a pasar desarrollar nuestro modelo de Machine Learning.

## Fase 2: Diseño y entrenamiento del modelo

Para este problema vamos a crear un modelo de regresión que tome como variables los datos climáticos y el índice del contaminante del día anterior y prediga el índice del contaminante del día siguiente. Para ello necesitaremos cruzar los datos, lo que podemos hacer basándonos en la fecha, para generar un único dataframe para el entrenamiento con todas las columnas relevantes.

Columnas que vamos a crear:
* Fecha: Columna que usaremos para unir los dataframes
* Temperatura media: Temperatura media de la ciudad en el día
* Temperatura mínima: Temperatura mínima de la ciudad en el día
* Temperatura máxima: Temperatura máxima de la ciudad en el día
* Precipitación: Precipitación en el día
* Velocidad del viento: Velocidad del viento en el día
* Presión: Presión atmosférica en el día
* Índice contaminante anterior: Indice del contaminante en el día anterior
* Índice contaminante: Indice del contaminante en el día (Columna objetivo)

De esta forma tendremos 6 datasets, uno por contaminante. Tendremos 6 modelos, uno para cada contaminante. Cada modelo tendrá como variables las columnas de temperatura, precipitación, velocidad del viento, presión y índice contaminante anterior. Esto podría solucionarse en otras librerías si se contuviera todos los datos de los índices de cada contaminante en un vector (un vector de 6 dimensiones, en este caso), sin embargo esto no es soportado en las funciones de MLib, requeriría crear un estimador personalizado o emplear otros frameworks, como TensorFlow.

In [32]:
# Vamos a unir los datos de los sensores con los datos del clima usando la fecha como clave

def unir_sensores_clima(datos_sensores: dict, df_clima: DataFrame) -> dict:
    """
    Une los datos de los sensores con los datos del clima usando la fecha como clave.
    """
    datos_completos = {}
    for key, sensor_df in datos_sensores.items():
        datos_completos[key] = sensor_df.join(df_clima, sensor_df.date == df_clima.fecha, how='inner')
        datos_completos[key] = datos_completos[key].drop(df_clima.fecha)
    return datos_completos

# Usamos la función para unir los datos de 2024
datos_completos = unir_sensores_clima(datos_2024_imputados, df_clima_imputado)

# Mostramos los datos completos de un sensor
datos_completos['viver'].show(10)

+----------+----+----+---+---+---+---+-----------------+------------------+------------------+-------------+--------------------+-------------------+
|      date|pm25|pm10| o3|no2|so2| co|temperatura media|temperatura mínima|temperatura máxima|precipitación|velocidad del viento|presión atmosférica|
+----------+----+----+---+---+---+---+-----------------+------------------+------------------+-------------+--------------------+-------------------+
|2024-01-01|  10|   8|  1|  1|  2|  0|             11.1|               9.8|              19.0|          0.0|                11.8|             1019.7|
|2024-01-02|  14|   2|  1|  1|  1|  0|             13.0|               7.8|              18.9|          0.0|                24.3|             1019.8|
|2024-01-03|   7|   7|  3|  3|  1|  0|             17.4|              15.0|              23.0|          0.0|                26.4|             1017.0|
|2024-01-04|  28|   2|  1|  1|  1|  0|             14.8|              14.5|              20.6|      

In [33]:
# Creamos una lista de los contaminantes que vamos a analizar
contaminantes = ['pm25', 'pm10', 'o3', 'no2', 'so2', 'co']

# Función para crear un diccionario de DataFrames por contaminante
def crear_datos_por_contaminante(datos_completos, contaminantes):
    """
    Crea un diccionario de DataFrames para cada contaminante y sensor.
    """
    datos_por_contaminante = {}
    for contaminante in contaminantes:
        datos_por_contaminante[contaminante] = {}
        for key, sensor_df in datos_completos.items():
            datos_por_contaminante[contaminante][key] = sensor_df.select(
                'date', contaminante, 'temperatura media', 'precipitación', 'velocidad del viento', 'presión atmosférica'
            )
    return datos_por_contaminante

# Creamos el diccionario usando la función
datos_por_contaminante = crear_datos_por_contaminante(datos_completos, contaminantes)
        
# Mostramos los datos de un contaminante para un sensor
datos_por_contaminante['pm25']['viver'].show(10)

+----------+----+-----------------+-------------+--------------------+-------------------+
|      date|pm25|temperatura media|precipitación|velocidad del viento|presión atmosférica|
+----------+----+-----------------+-------------+--------------------+-------------------+
|2024-01-01|  10|             11.1|          0.0|                11.8|             1019.7|
|2024-01-02|  14|             13.0|          0.0|                24.3|             1019.8|
|2024-01-03|   7|             17.4|          0.0|                26.4|             1017.0|
|2024-01-04|  28|             14.8|          0.6|                12.1|             1014.8|
|2024-01-05|   9|             13.1|          0.0|                22.7|             1006.0|
|2024-01-06|   7|             11.9|          0.0|                25.6|             1013.3|
|2024-01-07|  12|             10.8|          0.0|                30.0|             1018.7|
|2024-01-08|  17|              9.3|          0.0|                18.5|             1018.5|

A continuación, tenemos que editar cada dataframe para añadir la columna con el dato del índice del contaminante del día anterior, que vamos a utilizar como característica para el modelo. Para ello tendremos que imputar un valor para el día anteior al primero de los datos.

In [34]:
# Vamos a iterar sobre todos los dataframes para crear la columna definida previamente

def agregar_columna_prev(datos_por_contaminante):
    """
    Añade una columna con el valor del índice del contaminante del día anterior para cada DataFrame.
    Imputa 0 para el primer día disponible.
    Modifica el diccionario de entrada in-place y lo retorna.
    """
    for contaminante, sensor_dfs in datos_por_contaminante.items():
        for key, sensor_df in sensor_dfs.items():
            # Creamos la columna del día anterior
            sensor_df = sensor_df.withColumn(
                f'{contaminante}_prev',
                F.lag(F.col(contaminante), 1).over(Window.orderBy('date'))
            )
            # Imputamos 0 para el primer día
            first_date = sensor_df.select(F.min('date')).collect()[0][0]
            sensor_df = sensor_df.withColumn(
                f'{contaminante}_prev',
                F.when(F.col('date') == first_date, 0).otherwise(F.col(f'{contaminante}_prev'))
            )
            datos_por_contaminante[contaminante][key] = sensor_df
    return datos_por_contaminante

# Aplicamos la función
datos_por_contaminante = agregar_columna_prev(datos_por_contaminante)


In [35]:
# Mostramos un ejemplo de los datos de un contaminante con la columna del día anterior
datos_por_contaminante['pm25']['viver'].show(10)

+----------+----+-----------------+-------------+--------------------+-------------------+---------+
|      date|pm25|temperatura media|precipitación|velocidad del viento|presión atmosférica|pm25_prev|
+----------+----+-----------------+-------------+--------------------+-------------------+---------+
|2024-01-01|  10|             11.1|          0.0|                11.8|             1019.7|        0|
|2024-01-02|  14|             13.0|          0.0|                24.3|             1019.8|       10|
|2024-01-03|   7|             17.4|          0.0|                26.4|             1017.0|       14|
|2024-01-04|  28|             14.8|          0.6|                12.1|             1014.8|        7|
|2024-01-05|   9|             13.1|          0.0|                22.7|             1006.0|       28|
|2024-01-06|   7|             11.9|          0.0|                25.6|             1013.3|        9|
|2024-01-07|  12|             10.8|          0.0|                30.0|             1018.7| 

In [36]:
# Vamos a crear las transformaciones necesarias para el pipeline de Machine Learning
from pyspark.ml.feature import VectorAssembler

def crear_assemblers(contaminantes):
    """
    Crea un diccionario de VectorAssembler para cada contaminante.
    """
    assembler_dict = {}
    for contaminante in contaminantes:
        input_cols = [f'{contaminante}_prev', 'temperatura media', 'precipitación', 'velocidad del viento', 'presión atmosférica']
        assembler_dict[contaminante] = VectorAssembler(inputCols=input_cols, outputCol=f'{contaminante}_features')
    return assembler_dict

# Ejemplo de uso:
Assembler_dict = crear_assemblers(contaminantes)
print(Assembler_dict['pm25'].explainParams())

handleInvalid: How to handle invalid data (NULL and NaN values). Options are 'skip' (filter out rows with invalid data), 'error' (throw an error), or 'keep' (return relevant number of NaN in the output). Column lengths are taken from the size of ML Attribute Group, which can be set using `VectorSizeHint` in a pipeline before `VectorAssembler`. Column lengths can also be inferred from first rows of the data since it is safe to do so but only in case of 'error' or 'skip'). (default: error)
inputCols: input column names. (current: ['pm25_prev', 'temperatura media', 'precipitación', 'velocidad del viento', 'presión atmosférica'])
outputCol: output column name. (default: VectorAssembler_21f897a07124__output, current: pm25_features)


In [37]:
from pyspark.ml.regression import LinearRegression

# A continuación creamos los modelos para cada contaminante
def crear_modelos_lineales(contaminantes):
    """
    Crea un diccionario de modelos de regresión lineal para cada contaminante.
    """
    Model_dict = {}
    for contaminante in contaminantes:
        Model_dict[contaminante] = LinearRegression(
            featuresCol=f'{contaminante}_features',
            labelCol=contaminante,
            maxIter=20, regParam=0.3, elasticNetParam=0.5
        )
    return Model_dict

# Ejemplo de uso:
Model_dict = crear_modelos_lineales(contaminantes)

In [38]:
# Vamos a crear las pipelines para cada contaminante
from pyspark.ml import Pipeline

Pipeline_dict = {}
for contaminante in contaminantes:
    # Creamos una pipeline para cada contaminante
    Pipeline_dict[contaminante] = Pipeline(stages=[Assembler_dict[contaminante], Model_dict[contaminante]])

In [39]:
# Entrenamos los modelos para cada contaminante para todos los sensores
trained_models = {}
# Iteramos sobre cada contaminante
for contaminante in contaminantes:
    trained_models[contaminante] = {}  # Inicializamos el diccionario para cada contaminante
    for key, sensor_df in datos_por_contaminante[contaminante].items():
        # Entrenamos el modelo
        pipeline_model = Pipeline_dict[contaminante].fit(sensor_df)
        # Guardamos el modelo en el diccionario
        trained_models[contaminante][key] = pipeline_model
        # Mostramos un resumen del modelo
        print(f'Modelo entrenado para {contaminante} en {key}:')
        print(pipeline_model.stages[-1].summary)

Modelo entrenado para pm25 en viver:
Modelo entrenado para pm25 en moli del sol:
Modelo entrenado para pm25 en pista de silla:
Modelo entrenado para pm25 en politecnic:
Modelo entrenado para pm25 en quart de poblet:
Modelo entrenado para pm10 en viver:
Modelo entrenado para pm10 en moli del sol:
Modelo entrenado para pm10 en pista de silla:
Modelo entrenado para pm10 en politecnic:
Modelo entrenado para pm10 en quart de poblet:
Modelo entrenado para o3 en viver:
Modelo entrenado para o3 en moli del sol:
Modelo entrenado para o3 en pista de silla:
Modelo entrenado para o3 en politecnic:
Modelo entrenado para o3 en quart de poblet:
Modelo entrenado para no2 en viver:
Modelo entrenado para no2 en moli del sol:
Modelo entrenado para no2 en pista de silla:
Modelo entrenado para no2 en politecnic:
Modelo entrenado para no2 en quart de poblet:
Modelo entrenado para so2 en viver:
Modelo entrenado para so2 en moli del sol:
Modelo entrenado para so2 en pista de silla:
Modelo entrenado para so2 e

## Fase 3: Generación del streaming de datos

Con los modelos ya entrenados, vamos a generar el streaming de datos. Para ello, utilizaremos la librería Spark Streaming. Crearemos una fuente de datos seleccionando los datos preparados para test, realizando el mismo preprocesado que con los datos de test y después partiéndolos en ficheros más pequeños, que se irán leyendo uno a uno.

In [31]:
# Vamos a leer los datos de meteosat de 2025
datos_clima_2025 = 'C:\\Users\\demad\\Desktop\\Test\\DataScienceIEBS\\Bloque 7\\Spark\\Proyecto final\\Datos calidad aire\\meteosat_val_2025.csv'
df_clima_2025 = spark.read.csv(datos_clima_2025, header=True, inferSchema=True)

In [40]:
# Aplicamos las mismas transformaciones que hemos hecho con los datos de 2024
df_clima_2025 = ajustar_clima(df_clima_2025)

# Imputamos los valores nulos en los datos de clima de 2025
df_clima_2025 = imputar_valores(df_clima_2025, get_int=False, date_column='fecha')

# Verificamos que no tenemos datos nulos en los datos de clima de 2025
if df_clima_2025.filter(
    F.col('temperatura media').isNull() | 
    F.col('temperatura mínima').isNull() | 
    F.col('temperatura máxima').isNull() | 
    F.col('precipitación').isNull() | 
    F.col('velocidad del viento').isNull() | 
    F.col('presión atmosférica').isNull()
).count() > 0:
    print("Hay datos nulos en el DataFrame del clima de 2025.")
else:
    print("No hay datos nulos en el DataFrame del clima de 2025.")

No hay datos nulos en el DataFrame del clima de 2025.


In [41]:
def transformacion_sensores(datos_sensores, df_clima, contaminantes, año = 2025):
    """
    Aplica todas las transformaciones necesarias a los datos de los sensores para el año 2025:
    - Ajusta formato y tipos de datos
    - Filtra por año
    - Imputa valores nulos
    - Une con datos climáticos ya transformados e imputados
    """
    # Ajuste de formato y tipos
    datos_ajustados = {key: sensor_adjustments(df) for key, df in datos_sensores.items()}
    
    # Filtrado por año
    datos = {key: datos_anuales(df, año) for key, df in datos_ajustados.items()}
    
    # Imputación de valores nulos
    datos_imputados = {key: imputar_valores(df) for key, df in datos.items()}
    
    # Unión con datos climáticos
    datos_completos = unir_sensores_clima(datos_imputados, df_clima)

    # Separamos por contaminante
    datos_por_contaminante = crear_datos_por_contaminante(datos_completos, contaminantes)

    # Añadimos columna del día anterior
    datos_por_contaminante = agregar_columna_prev(datos_por_contaminante)
    
    return datos_por_contaminante

# Ejecutamos la transformación de los sensores para el año 2025
datos_completos_2025 = transformacion_sensores(datos_sensores, df_clima_2025, contaminantes)

In [42]:
# Verificamos los datos transformados de un sensor
datos_completos_2025['pm25']['viver'].show(10)


+----------+----+-----------------+-------------+--------------------+-------------------+---------+
|      date|pm25|temperatura media|precipitación|velocidad del viento|presión atmosférica|pm25_prev|
+----------+----+-----------------+-------------+--------------------+-------------------+---------+
|2025-01-01|  29|              9.4|          3.8|                11.8|             1030.2|        0|
|2025-01-02|   9|              9.2|          0.0|                13.1|             1027.0|       29|
|2025-01-03|   6|             12.1|          0.0|                17.7|             1022.8|        9|
|2025-01-04|  14|             13.0|          0.0|                19.9|             1020.7|        6|
|2025-01-05|  16|             12.8|          0.0|                19.4|             1014.0|       14|
|2025-01-06|  13|             13.5|          0.0|                16.6|             1007.7|       16|
|2025-01-07|  13|             11.9|          0.0|                20.5|             1018.3| 

In [ ]:
# Vamos a partir los datos de modo que generemos una fuente de streaming, con un dato por día y por sensor
def generar_streaming(datos_completos, savepath):
    """
    Genera un DataFrame de streaming con un dato por día y por sensor.
    """
    import os
    if not os.path.exists(savepath):
        os.makedirs(savepath)
    # Generamos un CSV por día y por sensor
    for key, sensor_df in datos_completos.items():
        # Generamos un directorio para cada sensor
        sensor_dir = os.path.join(savepath, key)
        if not os.path.exists(sensor_dir):
            os.makedirs(sensor_dir)
            
        # Generamos una lista de fechas únicas
        fechas = [row['date'] for row in sensor_df.select('date').distinct().collect()]
        
        # Iteramos sobre las fechas y guardamos un CSV por día
        for fecha in fechas:
            df_dia = sensor_df.filter(F.col('date') == fecha)
            fecha_str = fecha.strftime('%Y-%m-%d')  # Convertimos la fecha a string
            output_path = os.path.join(sensor_dir, fecha_str)
            df_dia.coalesce(1).write.format("csv").option("header", True).save(output_path)

El código siguiente tarda bastante en ejecutarse, pero puede ejecutarse una única vez para generar las carpetas y los datos separados. Se ha copiado más adelante la ruta porque no es necesario volver a ejecutarlo entre sesiones si los datos ya existen.

In [58]:
# Vamos a generar los datos para su posterior lectura en streaming
savepath = 'C:\\Users\\demad\\Desktop\\Test\\DataScienceIEBS\\Bloque 7\\Spark\\Proyecto final\\Datos calidad aire\\Datos streaming'

# Llamamos a la función para cada contaminante
for contaminante in contaminantes:
    # Creamos un directorio para cada contaminante
    contaminante_dir = os.path.join(savepath, contaminante)
    os.makedirs(contaminante_dir, exist_ok=True)
    
    # Generamos el streaming para el contaminante
    generar_streaming(datos_completos_2025[contaminante], contaminante_dir)

Ahora hemos generado una carpeta para cada contaminante, en las cuales tenemos los datos por cada sensor, y en ellos un CSV con los datos diarios. La siguiente parte realizar la ingesta de estos datos en Streaming, empleando para ello la librería de Spark Streaming.

## Fase 4: Lectura y procesamiento de datos en streaming

A continuación vamos a implementar el flujo de streaming apropiado para cada uno de los contaminantes, de modo que se lean los datos diarios y se sirvan a la pipeline con el modelo entrenado previamente, generando las sucesivas predicciones. 

A partir de los datos almacenados en el paso anterior, vamos a emplear el método readStream para generar un flujo de datos por cada contaminante y sensor. Cada flujo de datos se alimentará a la pipeline con el modelo entrenado previamente, generando así las predicciones

In [43]:
# Definimos el esquema de los datos que vamos a leer en streaming

# Creamos un diccionario para almacenar los esquemas de cada contaminante
schemas = {}

# Generamos el esquema para cada contaminante
for contaminante in contaminantes:
    schemas[contaminante] = StructType([
        StructField('date', DateType(), True),
        StructField(f'{contaminante}', IntegerType(), True),
        StructField('temperatura media', FloatType(), True),
        StructField('precipitación', FloatType(), True),
        StructField('velocidad del viento', FloatType(), True),
        StructField('presión atmosférica', FloatType(), True),
        StructField(f'{contaminante}_prev', IntegerType(), True)
    ])

In [ ]:
# Creamos los flujos de datos para cada contaminante y sensor

flujos = []
savepath = 'C:\\Users\\demad\\Desktop\\Test\\DataScienceIEBS\\Bloque 7\\Spark\\Proyecto final\\Datos calidad aire\\Datos streaming'


for contaminante in contaminantes:
    # Definimos la ruta de los datos para el contaminante
    contaminante_path = os.path.join(savepath, contaminante)
    
    for key in datos_completos_2025[contaminante].keys(): # type: ignore
        # Definimos la ruta de los datos para el sensor
        sensor_path = os.path.join(contaminante_path, key, '*', '*.csv')
        
        # Leemos los datos del sensor en streaming
        SourceStream = spark.readStream \
        .format('csv') \
        .option('header', True) \
        .schema(schemas[contaminante]) \
        .option('ignoreLeadingWhiteSpace', True) \
        .option('mode', 'dropMalformed') \
        .load(sensor_path)
        
        # Aplicamos el modelo previamente entrenado para el contaminante y sensor específicos
        stream_predictions = trained_models[contaminante][key].transform(SourceStream)
        
        # Generamos el nombre para la query
        query_name = f'{contaminante}_{key}_query'.replace(' ', '_')
        
        # Escribimos la salida del modelo en memoria
        query = stream_predictions.writeStream \
            .outputMode('append') \
            .format('memory') \
            .queryName(query_name) \
            .start()
            
        flujos.append(query)

In [ ]:
# Vamos a consultar las predicciones con Spark SQL
resultados = {}
query_names = []
for contaminante in contaminantes:
    for key in datos_completos_2025[contaminante].keys():  # type: ignore
        query_name = f'{contaminante}_{key}_query'.replace(' ', '_')
        resultados[query_name] = spark.sql(f'SELECT * FROM {query_name} ORDER BY date ASC')
        query_names.append(query_name)
        
# Mostramos los resultados de las predicciones para el primer contaminante y sensor
resultados['pm25_politecnic_query'].show(10)

+----------+----+-----------------+-------------+--------------------+-------------------+---------+--------------------+------------------+
|      date|pm25|temperatura media|precipitación|velocidad del viento|presión atmosférica|pm25_prev|       pm25_features|        prediction|
+----------+----+-----------------+-------------+--------------------+-------------------+---------+--------------------+------------------+
|2025-01-01|  53|              9.4|          3.8|                11.8|             1030.2|        0|[0.0,9.3999996185...|16.303287666211354|
|2025-01-02|  39|              9.2|          0.0|                13.1|             1027.0|       53|[53.0,9.199999809...| 46.47274900577915|
|2025-01-03|  31|             12.1|          0.0|                17.7|             1022.8|       39|[39.0,12.10000038...|34.865155281058335|
|2025-01-04|  34|             13.0|          0.0|                19.9|             1020.7|       31|[31.0,13.0,0.0,19...|28.382518537226638|
|2025-01-05| 

In [ ]:
# Revisamos el estado de la query de streaming
for query in flujos:
    print(f"Query: {query.name}, Status: {query.status}, Last Progress: {query.lastProgress}")

Query: pm25_viver_query, Status: {'message': 'Getting offsets from FileStreamSource[file:/C:/Users/demad/Desktop/Test/DataScienceIEBS/Bloque 7/Spark/Proyecto final/Datos calidad aire/Datos streaming/pm25/viver/*/*.csv]', 'isDataAvailable': False, 'isTriggerActive': True}, Last Progress: {'id': '785148c7-2c97-4885-9888-3ad37f26d8ac', 'runId': 'c118ad8f-300e-438c-a353-8e3f52fa342a', 'name': 'pm25_viver_query', 'timestamp': '2025-07-02T10:11:31.124Z', 'batchId': 1, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0, 'durationMs': {'latestOffset': 44671, 'triggerExecution': 44671}, 'stateOperators': [], 'sources': [{'description': 'FileStreamSource[file:/C:/Users/demad/Desktop/Test/DataScienceIEBS/Bloque 7/Spark/Proyecto final/Datos calidad aire/Datos streaming/pm25/viver/*/*.csv]', 'startOffset': {'logOffset': 0}, 'endOffset': {'logOffset': 0}, 'latestOffset': None, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0}], 'sink': {'descriptio

Con los resultados anteriores vemos que Spark ha terminado de procesar todos los datos servidos en streaming y que no están apareciendo datos nuevos. Si se conectase el Sink a una API que pudiera servirle datos de forma continua, esta pipeline continuaría leyendo y procesando datos en tiempo real.

## Fase 5: Evaluación del modelo

Con las predicciones realizadas en cada una de las querys para todos los contaminantes y sensores disponibles, vamos a pasar a valorar los resultados obtenidos. Dado que se trata de una regresión lineal, vamos a emplear las siguientes métricas de evaluación:

* RMSE - Root Mean Squared Error (Error cuadrático medio)
* MAE - Mean Absolute Error (Error absoluto medio)
* R2 - Coefficient of Determination (Coeficiente de determinación)
* MAPE - Mean Absolute Percentage Error (Error absoluto porcentual medio)

In [ ]:
# Vamos a evaluar los modelos de regresión lineal para cada contaminante y sensor
from pyspark.ml.evaluation import RegressionEvaluator

def evaluar_modelos(resultados, query_names, contaminantes):
    """
    Evalúa los modelos de regresión lineal para cada contaminante y sensor.
    Devuelve un diccionario con las métricas RMSE, MAE, R2 y MAPE para cada query.
    """
    metricas = {}
    for query_name in query_names:
        # Detectar contaminante a partir del nombre de la query
        contaminante_actual = next((c for c in contaminantes if query_name.startswith(c)), None)
        if contaminante_actual is None:
            continue
        resultados_df = resultados[query_name]

        # RMSE
        evaluator_rmse = RegressionEvaluator(
            labelCol=contaminante_actual,
            predictionCol='prediction',
            metricName='rmse'
        )
        # MAE
        evaluator_mae = RegressionEvaluator(
            labelCol=contaminante_actual,
            predictionCol='prediction',
            metricName='mae'
        )
        # R2
        evaluator_r2 = RegressionEvaluator(
            labelCol=contaminante_actual,
            predictionCol='prediction',
            metricName='r2'
        )

        # Calculamos las métricas
        rmse = evaluator_rmse.evaluate(resultados_df)
        mae = evaluator_mae.evaluate(resultados_df)
        r2 = evaluator_r2.evaluate(resultados_df)

        # MAPE manual
        mape_df = resultados_df.withColumn(
            "ape",
            (F.abs(F.col(contaminante_actual) - F.col("prediction")) / F.when(F.col(contaminante_actual) != 0, F.abs(F.col(contaminante_actual))).otherwise(None))
        )
        mape = mape_df.select(F.mean("ape")).collect()[0][0]
        mape = mape * 100 if mape is not None else None

        metricas[query_name] = {
            'contaminante': contaminante_actual,
            'rmse': rmse,
            'mae': mae,
            'r2': r2,
            'mape': mape
        }
    return metricas

RMSE para pm25 en pm25_viver_query: 8.915952065948117
MAE para pm25 en pm25_viver_query: 6.4807179232418335
R2 para pm25 en pm25_viver_query: 0.4956864381145868
MAPE para pm25 en pm25_viver_query: 45.24018015216404
RMSE para pm25 en pm25_moli_del_sol_query: 14.209043016283905
MAE para pm25 en pm25_moli_del_sol_query: 9.966120213101641
R2 para pm25 en pm25_moli_del_sol_query: 0.5573353672495425
MAPE para pm25 en pm25_moli_del_sol_query: 36.98056007435097


In [ ]:
# Evaluamos los modelos
metricas = evaluar_modelos(resultados, query_names, contaminantes)

# Mostramos las métricas de los modelos
for query_name, metrics in metricas.items():
    print(f"Query: {query_name}, Contaminante: {metrics['contaminante']}, RMSE: {metrics['rmse']:.2f}, MAE: {metrics['mae']:.2f}, R2: {metrics['r2']:.2f}, MAPE: {metrics['mape']:.2f}%")
    print("--------------------------------------------------------")